# HW4 - Notebook 2: Model validation (2010–2015 only)

Compare candidate models on the development period with chronological folds, then freeze one model before the test.

Code: [`model.py`](model.py).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import sys

# Course setup
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))

from features import PANEL_PATH
from model import development_rows, validate, summarize, FACTOR_SIGNS, FINAL_MODEL

panel = pd.read_parquet(PANEL_PATH)

## 1. Data errors and the investability screening

Some extreme 90-day returns are vendor split/unit errors (e.g. AAM: +10 million %). I keep only stocks that are tradeable
at the signal close: raw close ≥ \$5 and median 63-day dollar volume ≥ \$1M. For evaluation only, returns are capped at +200%.

In [ ]:
dev_all = panel.loc[panel.date.between('2010-01-01', '2015-12-31')]
# screened 2010-2015 rows
dev = development_rows(panel)
# before & after screening
print(f'Rows with |90-day return| > 500%: {(dev_all.target.abs() > 5).sum()} before the screen, '
      f'{(dev.target.abs() > 5).sum()} after')
print(f'Screened rows: {len(dev):,} of {len(dev_all):,}')

Rows with |90-day return| > 500%: 24 before the screen, 5 after
Screened rows: 106,583 of 141,562


## 2. Chronological folds

Expanding window, one validation year at a time (2012–2015). Training uses only rows whose 90-day outcome finished
before the first validation signal.

In [ ]:
# folds only
_, folds = validate(dev, candidates={})
display(pd.DataFrame([{
    'validation_year': year, 'training_rows': len(tr), 'validation_rows': len(va),
    'last_training_outcome': dev.iloc[tr].label_end.max().date(),
    'first_validation_signal': dev.iloc[va].date.min().date()}
    for year, (tr, va) in zip([2012, 2013, 2014, 2015], folds)]))
assert all(dev.iloc[tr].label_end.max() < dev.iloc[va].date.min() for tr, va in folds)

,validation_year,training_rows,validation_rows,last_training_outcome,first_validation_signal
0,2012,28553,16207,2012-01-30,2012-01-31
1,2013,44701,17902,2013-01-30,2013-01-31
2,2014,62195,20036,2014-01-30,2014-01-31
3,2015,80272,21195,2014-12-30,2015-01-30


## 3. Compare candidates

Metrics per fold: 
- rank IC
- top-30 excess return
- top-decile minus bottom-decile return.

In [ ]:
results, _ = validate(dev)
summary = summarize(results)
display(summary.round(4))
display(results.pivot(index='model', columns='validation_year', values='rank_IC').loc[summary.index].round(3))

,mean_rank_IC,IC_std,folds_IC_positive,mean_top30_excess,mean_decile_spread
model,,,,,
"Factor score (fixed weights, no fitting)",0.0622,0.0861,3/4,0.0135,0.0263
"Boosted trees, rank features, excess target",-0.0042,0.0121,2/4,0.0033,-0.0017
"Boosted trees, raw features, excess target",-0.0071,0.0233,1/4,-0.0009,-0.0044
"Ridge, raw features, raw target (course baseline)",-0.0191,0.0163,1/4,-0.0129,-0.0116
"Ridge (alpha=1000), rank features, excess target",-0.0360,0.0196,0/4,-0.0155,-0.0193
"Ridge, rank features, excess target",-0.0360,0.0192,0/4,-0.0157,-0.0191
"Ridge, 5 factor ranks, excess target",-0.0488,0.0427,0/4,-0.0099,-0.0287


validation_year,2012,2013,2014,2015
model,,,,
"Factor score (fixed weights, no fitting)",0.038,-0.042,0.092,0.161
"Boosted trees, rank features, excess target",-0.010,-0.019,0.006,0.006
"Boosted trees, raw features, excess target",0.024,-0.004,-0.021,-0.028
"Ridge, raw features, raw target (course baseline)",-0.019,-0.038,0.002,-0.022
"Ridge (alpha=1000), rank features, excess target",-0.030,-0.042,-0.013,-0.059
"Ridge, rank features, excess target",-0.030,-0.042,-0.013,-0.058
"Ridge, 5 factor ranks, excess target",-0.020,-0.044,-0.021,-0.111


## 4. Decision

**Model: fixed-weight factor score**, the average of five signals in the same-date ranks. Each month buy the top 30.

In [5]:
print('Frozen model:', FINAL_MODEL)
print('Signals (+1 higher is better, -1 lower is better):', FACTOR_SIGNS)

Frozen model: Factor score (fixed weights, no fitting)
Signals (+1 higher is better, -1 lower is better): {'rank_operating_roa': 1, 'rank_fcf_to_assets': 1, 'rank_drawdown_252': 1, 'rank_momentum_252_21': 1, 'rank_volatility_63': -1}


- It is the only candidate with a positive mean rank IC (about +0.06, positive in 3 of 4 years).
- Every fitted model had a negative IC including Ridge on the same five features. Since Signal strength flips between
  years, fixed weights avoid chasing the last years compared to the fitted weights.
- No retraining in the test: the weights are fixed.